In [1]:
import sqlite3
import pandas as pd
import os

db_path = '../data/hidrocarburos.db'

# Eliminar la base de datos si ya existe para empezar limpio
if os.path.exists(db_path):
    os.remove(db_path)
    print('Base de datos anterior eliminada.')

# Crear nueva conexión
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Crear esquema
cursor.executescript('''
    CREATE TABLE IF NOT EXISTS ubicaciones (
        id_ubicacion    INTEGER PRIMARY KEY AUTOINCREMENT,
        estado_pais     TEXT NOT NULL,
        nombre_campo    TEXT
    );

    CREATE TABLE IF NOT EXISTS muestras (
        id_muestra      INTEGER PRIMARY KEY AUTOINCREMENT,
        id_ubicacion    INTEGER,
        tipo_crudo      TEXT NOT NULL,
        clase_azufre    TEXT NOT NULL,
        FOREIGN KEY (id_ubicacion) REFERENCES ubicaciones(id_ubicacion)
    );

    CREATE TABLE IF NOT EXISTS propiedades (
        id_propiedad    INTEGER PRIMARY KEY AUTOINCREMENT,
        id_muestra      INTEGER NOT NULL,
        api_crude       REAL,
        azufre_pct      REAL,
        nitrogeno_pct   REAL,
        viscosidad_100f REAL,
        punto_fluidez   REAL,
        residuo_carbono REAL,
        vol_gasolina    REAL,
        vol_nafta       REAL,
        vol_residuo     REAL,
        FOREIGN KEY (id_muestra) REFERENCES muestras(id_muestra)
    );
''')

conn.commit()
print('Base de datos creada con esquema limpio.')
print('Tablas: ubicaciones, muestras, propiedades')

Base de datos anterior eliminada.
Base de datos creada con esquema limpio.
Tablas: ubicaciones, muestras, propiedades


In [2]:
# Cargar dataset limpio
df = pd.read_csv('../data/crude_oil_clean.csv')
df['DULCE_AGRIO'] = df['SRC'].apply(
    lambda x: 'Dulce (sweet)' if x < 0.5 else 'Agrio (sour)'
)

print(f'Cargando {len(df):,} muestras en la base de datos...')

# ── 1. Insertar ubicaciones únicas ───────────────────────────
ubicaciones = df[['STCTNAME', 'FIELD_NAME']].drop_duplicates().reset_index(drop=True)
ubicaciones.columns = ['estado_pais', 'nombre_campo']
ubicaciones.to_sql('ubicaciones', conn, if_exists='append', index=False)
print(f'Ubicaciones insertadas: {len(ubicaciones):,}')

# ── 2. Insertar muestras ─────────────────────────────────────
# Obtener id_ubicacion para cada muestra
query_ubic = 'SELECT id_ubicacion, estado_pais, nombre_campo FROM ubicaciones'
df_ubic = pd.read_sql(query_ubic, conn)

df_merged = df.merge(
    df_ubic,
    left_on=['STCTNAME', 'FIELD_NAME'],
    right_on=['estado_pais', 'nombre_campo'],
    how='left'
)

muestras = df_merged[['id_ubicacion', 'TIPO_CRUDO', 'DULCE_AGRIO']].copy()
muestras.columns = ['id_ubicacion', 'tipo_crudo', 'clase_azufre']
muestras.to_sql('muestras', conn, if_exists='append', index=False)
print(f'Muestras insertadas: {len(muestras):,}')

# ── 3. Insertar propiedades ──────────────────────────────────
# Obtener id_muestra generados
df_muestras_db = pd.read_sql('SELECT id_muestra FROM muestras', conn)

propiedades = pd.DataFrame({
    'id_muestra':      df_muestras_db['id_muestra'],
    'api_crude':       df['API_CRUDE'].values,
    'azufre_pct':      df['SRC'].values,
    'nitrogeno_pct':   df['CRN'].values,
    'viscosidad_100f': df['SU100'].values,
    'punto_fluidez':   df['POUR_POINT'].values,
    'residuo_carbono': df['CAR_CR_WT'].values,
    'vol_gasolina':    df['LT_GAS_VOL'].values,
    'vol_nafta':       df['GAS_NP_VOL'].values,
    'vol_residuo':     df['RESDUM_VOL'].values,
})
propiedades.to_sql('propiedades', conn, if_exists='append', index=False)
print(f'Propiedades insertadas: {len(propiedades):,}')

conn.commit()
print('\nBase de datos cargada correctamente.')

Cargando 8,842 muestras en la base de datos...
Ubicaciones insertadas: 4,007
Muestras insertadas: 8,842
Propiedades insertadas: 8,842

Base de datos cargada correctamente.


In [3]:
# ── CONSULTAS SQL DE NEGOCIO ─────────────────────────────────

print('=' * 60)
print('CONSULTA 1: Distribución de tipos de crudo por origen')
print('=' * 60)
q1 = '''
SELECT 
    u.estado_pais,
    m.tipo_crudo,
    COUNT(*) AS cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY u.estado_pais), 1) AS porcentaje
FROM muestras m
JOIN ubicaciones u ON m.id_ubicacion = u.id_ubicacion
GROUP BY u.estado_pais, m.tipo_crudo
HAVING COUNT(*) > 50
ORDER BY u.estado_pais, cantidad DESC
LIMIT 20
'''
print(pd.read_sql(q1, conn).to_string(index=False))

print('\n' + '=' * 60)
print('CONSULTA 2: Propiedades promedio por tipo de crudo')
print('=' * 60)
q2 = '''
SELECT
    m.tipo_crudo,
    COUNT(*)                          AS muestras,
    ROUND(AVG(p.api_crude), 1)        AS api_promedio,
    ROUND(AVG(p.azufre_pct), 3)       AS azufre_prom_pct,
    ROUND(AVG(p.viscosidad_100f), 1)  AS viscosidad_prom,
    ROUND(AVG(p.residuo_carbono), 2)  AS residuo_carbono_prom,
    ROUND(AVG(p.vol_nafta), 1)        AS vol_nafta_prom
FROM muestras m
JOIN propiedades p ON m.id_muestra = p.id_muestra
GROUP BY m.tipo_crudo
ORDER BY api_promedio DESC
'''
print(pd.read_sql(q2, conn).to_string(index=False))

print('\n' + '=' * 60)
print('CONSULTA 3: ¿Qué % de crudos agrios hay por tipo?')
print('=' * 60)
q3 = '''
SELECT
    m.tipo_crudo,
    COUNT(*) AS total,
    SUM(CASE WHEN m.clase_azufre = 'Agrio (sour)' THEN 1 ELSE 0 END) AS agrios,
    ROUND(
        SUM(CASE WHEN m.clase_azufre = 'Agrio (sour)' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1
    ) AS pct_agrio
FROM muestras m
GROUP BY m.tipo_crudo
ORDER BY pct_agrio DESC
'''
print(pd.read_sql(q3, conn).to_string(index=False))

print('\n' + '=' * 60)
print('CONSULTA 4: Top 10 campos con mayor azufre promedio')
print('=' * 60)
q4 = '''
SELECT
    u.estado_pais,
    u.nombre_campo,
    COUNT(*)                    AS muestras,
    ROUND(AVG(p.azufre_pct), 3) AS azufre_promedio_pct,
    ROUND(AVG(p.api_crude), 1)  AS api_promedio
FROM propiedades p
JOIN muestras m    ON p.id_muestra   = m.id_muestra
JOIN ubicaciones u ON m.id_ubicacion = u.id_ubicacion
GROUP BY u.estado_pais, u.nombre_campo
HAVING COUNT(*) >= 5
ORDER BY azufre_promedio_pct DESC
LIMIT 10
'''
print(pd.read_sql(q4, conn).to_string(index=False))

print('\n' + '=' * 60)
print('CONSULTA 5: Crudos con API > 40 y azufre > 1% (atípicos)')
print('(Livianos pero agrios — interesantes para refinería)')
print('='* 60)
q5 = '''
SELECT
    u.estado_pais,
    u.nombre_campo,
    ROUND(p.api_crude, 1)  AS api,
    ROUND(p.azufre_pct, 3) AS azufre_pct,
    ROUND(p.viscosidad_100f, 1) AS viscosidad
FROM propiedades p
JOIN muestras m    ON p.id_muestra   = m.id_muestra
JOIN ubicaciones u ON m.id_ubicacion = u.id_ubicacion
WHERE p.api_crude > 40 AND p.azufre_pct > 1.0
ORDER BY p.azufre_pct DESC
LIMIT 10
'''
print(pd.read_sql(q5, conn).to_string(index=False))

conn.close()
print('\nConexión cerrada.')

CONSULTA 1: Distribución de tipos de crudo por origen
 estado_pais tipo_crudo  cantidad  porcentaje
    ARKANSAS    Liviano        98       100.0
  CALIFORNIA    Liviano       142        37.7
  CALIFORNIA    Mediano       136        36.1
  CALIFORNIA     Pesado        99        26.3
    COLORADO    Liviano       244       100.0
    ILLINOIS    Liviano       135       100.0
      KANSAS    Liviano       476        81.8
      KANSAS    Mediano       106        18.2
   LOUISIANA    Liviano       527        81.7
   LOUISIANA    Mediano       118        18.3
    MICHIGAN    Liviano       110       100.0
 MISSISSIPPI    Liviano        91       100.0
     MONTANA    Liviano       121       100.0
    NEBRASKA    Liviano        83       100.0
  NEW MEXICO    Liviano       219       100.0
NORTH DAKOTA    Liviano       107       100.0
        OHIO    Liviano        56       100.0
    OKLAHOMA    Liviano      1169        92.2
    OKLAHOMA    Mediano        99         7.8
PENNSYLVANIA    Liviano   